# Categorical Variables

In this section we will be using categorical variables in our regression. We will be looking at the Ford Focus dataset and use the transmission type as a categorical variable. We will construct our design matrix using two dummy variables for the three transmission types: automatic, manual, and semi-automatic.

Run the code below to create the design matrix with the dummy varaibles and to fit the model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm

In [ ]:
data = pd.read_csv("data/Focus.csv")

# Extracting variables for regression
data = data[["mileage", "transmission", "price"]]
prices = data["price"]
mileages = data["mileage"]
transmission = data["transmission"]

# Creating dummy variables for transmission
dummy_vars = pd.get_dummies(data["transmission"], prefix="trans")
comparison = pd.concat([data["transmission"], dummy_vars], axis=1).head(10)
print(comparison.head())

# Constructing the design matrix with dummy variables
X_with_dummies = pd.get_dummies(
    data[["mileage", "transmission"]], columns=["transmission"], drop_first=True
)
X_design = sm.add_constant(X_with_dummies)
print(f"\nDesign matrix:\n{X_design.head()}")

Now that we have the design matrix we can fit the model. Run the code below to see the summary of the model.

In [ ]:
model = sm.OLS(prices, X_design)
results = model.fit()
results.summary()

The model can be visualised using a scatterplot and all three lines for the categorical variables. Run the code below to plot regression lines and the data sorted by transmission type. Observe how they have the same slope.

In [ ]:
colors = {"Manual": "blue", "Automatic": "red", "Semi-Auto": "green"}
symbols = {"Manual": "o", "Automatic": "s", "Semi-Auto": "^"}
transmission_types = ["Manual", "Automatic", "Semi-Auto"]
coeffs = results.params

plt.figure(figsize=(16, 12))

# Plot data points
for trans_type in transmission_types:
    mask = data["transmission"] == trans_type
    plt.scatter(
        data.loc[mask, "mileage"],
        data.loc[mask, "price"],
        c=colors[trans_type],
        marker=symbols[trans_type],
        label=trans_type,
        alpha=0.7,
        s=30,
    )

# Add regression lines
mileage_range = np.linspace(data["mileage"].min(), data["mileage"].max(), 100)
for trans_type in transmission_types:
    if trans_type == "Automatic":
        intercept = coeffs["const"]
    else:
        intercept = coeffs["const"] + coeffs[f"transmission_{trans_type}"]

    price_line = intercept + coeffs["mileage"] * mileage_range
    plt.plot(
        mileage_range,
        price_line,
        color=colors[trans_type],
        linewidth=3,
        linestyle="--",
        alpha=0.8,
    )

plt.xlabel("Mileage")
plt.ylabel("Price")
plt.title("Price vs Mileage by Transmission Type")
plt.legend()
plt.grid(True, alpha=0.3)